# Phase 2 - The Dataset and the Classical Baseline

Before testing any quantum model, we need two things:

1. A **dataset** that is hard enough that the encoding choice matters.
2. A **classical baseline** to compare against.

A baseline is essential. Without it, a quantum accuracy of (say) 85% is
meaningless - we cannot tell whether it is good or bad. The classical
**support vector machine (SVM)** with an RBF kernel is a strong, standard
classifier and is our yardstick for the rest of the project.

In [ ]:
"""02_datasets_and_classical_baseline.ipynb"""

# Cell 01 - Imports and reproducible seed

import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC

import qml_utils as hu

np.random.seed(hu.RANDOM_SEED)
print(f"Random seed: {hu.RANDOM_SEED}")

---
**Cell 02 - Load and view the two-moons dataset.**
The two-moons dataset is two interleaving half circles. It cannot be
separated by a straight line, so a model must learn a curved boundary. It is
also two-dimensional, which lets us draw decision boundaries later.

In [ ]:
# Cell 02 - Generate the dataset and plot the raw points

import matplotlib.pyplot as plt

x_train, x_test, y_train, y_test = hu.load_moons(n_samples=200, noise=0.20)

print(f"Training samples: {x_train.shape[0]}")
print(f"Test samples    : {x_test.shape[0]}")
print(f"Features each   : {x_train.shape[1]}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
hu.plot_dataset(x_train, y_train, title="Training set (raw)", ax=ax1)
hu.plot_dataset(x_test, y_test, title="Test set (raw)", ax=ax2)
plt.tight_layout()
plt.show()

---
**Cell 03 - Scale the features.**
The quantum encodings interpret feature values as rotation angles, so we
rescale every feature into the range $[0, \pi]$. The scaler is fit on the
**training set only** and then applied to the test set, which prevents
information from the test set leaking into preprocessing. We use the same
scaled data for the classical baseline so the comparison is fair.

In [ ]:
# Cell 03 - Scale features to [0, pi] using training statistics only

x_train_scaled, x_test_scaled, scaler = hu.scale_features(
    x_train, x_test, feature_range=(0.0, np.pi)
)

print("Training feature ranges after scaling:")
print(
    f"  feature 1: [{x_train_scaled[:, 0].min():.3f}, {x_train_scaled[:, 0].max():.3f}]"
)
print(
    f"  feature 2: [{x_train_scaled[:, 1].min():.3f}, {x_train_scaled[:, 1].max():.3f}]"
)

---
**Cell 04 - Train the classical RBF-kernel SVM.**
The radial basis function (RBF) kernel lets the SVM draw smooth, curved
decision boundaries, which suits the two-moons shape. We report accuracy on
both the training and test sets. The **test accuracy** is the number that
matters: it measures how well the model generalizes to data it has not seen.

In [ ]:
# Cell 04 - Fit the RBF SVM and report accuracy

svm = SVC(kernel="rbf", gamma="scale", random_state=hu.RANDOM_SEED)
svm.fit(x_train_scaled, y_train)

svm_train_acc = accuracy_score(y_train, svm.predict(x_train_scaled))
svm_test_acc = accuracy_score(y_test, svm.predict(x_test_scaled))

print(f"RBF SVM training accuracy: {svm_train_acc:.3f}")
print(f"RBF SVM test accuracy    : {svm_test_acc:.3f}")

---
**Cell 05 - Visualize the SVM decision boundary.**
The colored regions show how the SVM partitions the feature space. The points
are the actual data. A good model wraps its boundary cleanly around the two
moons. This is the picture every quantum model will be compared against.

In [ ]:
# Cell 05 - Plot the SVM decision boundary on the test set

hu.plot_decision_boundary(
    svm.predict,
    x_test_scaled,
    y_test,
    title=f"RBF SVM (test accuracy {svm_test_acc:.2f})",
)
plt.show()

---
**Cell 06 - A linear baseline for contrast.**
To show why a curved boundary is needed, we also fit a **linear** SVM. Its
straight boundary cannot follow the moons, so its accuracy is lower. This
makes the point that the *shape* a model can draw, which the encoding choice
controls in the quantum case, directly affects performance.

In [ ]:
# Cell 06 - Linear SVM for comparison

linear_svm = SVC(kernel="linear", random_state=hu.RANDOM_SEED)
linear_svm.fit(x_train_scaled, y_train)
linear_test_acc = accuracy_score(y_test, linear_svm.predict(x_test_scaled))
print(f"Linear SVM test accuracy: {linear_test_acc:.3f}")

hu.plot_decision_boundary(
    linear_svm.predict,
    x_test_scaled,
    y_test,
    title=f"Linear SVM (test accuracy {linear_test_acc:.2f})",
)
plt.show()

---
## Record the benchmark

Write down the **RBF SVM test accuracy** printed in Cell 04. That is the
number every quantum encoding must be measured against in Phase 7. According
to our hypothesis, we do not expect any quantum encoding to beat it on this
small, clean dataset, and that is a perfectly acceptable result.